In [ ]:
#!pip install -q langchain-openai langchain-core requests langgraph -q

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

In [2]:
# --- Define the state ---
class State(TypedDict):
    user_input: str
    draft_reply: str
    approved_reply: str

In [3]:
# --- Node: AI drafts a response ---
def draft_response(state: State) -> State:
    user_query = state["user_input"]
    # Dummy banking logic
    if "transfer" in user_query and "10000" in user_query:
        state["draft_reply"] = (
            "You requested a transfer of $10,000. "
            "This requires compliance approval before processing."
        )
    else:
        state["draft_reply"] = f"I can help with your request: {user_query}"
    return state

In [4]:
# --- Node: Human review (Human-in-the-loop) ---
def human_review(state: State) -> State:
    print("\n--- HUMAN APPROVAL NEEDED ---")
    print("AI draft reply:", state["draft_reply"])

    approval = input("Approve this reply? (y/n): ").strip().lower()

    if approval == "y":
        state["approved_reply"] = state["draft_reply"]
    else:
        state["approved_reply"] = "A human agent will handle your request."
    return state

In [5]:
# --- Build the graph ---
workflow = StateGraph(State)
workflow.add_node("draft", draft_response)
workflow.add_node("review", human_review)

workflow.set_entry_point("draft")
workflow.add_edge("draft", "review")
workflow.add_edge("review", END)

graph = workflow.compile()

In [9]:
# Example 1: High-value transfer (needs approval)
user_question = "I want to transfer $10000 to my savings account."
final_state = graph.invoke({"user_input": user_question})
print("\n--- FINAL OUTPUT ---")
print(final_state["approved_reply"])


--- HUMAN APPROVAL NEEDED ---
AI draft reply: You requested a transfer of $10,000. This requires compliance approval before processing.

--- FINAL OUTPUT ---
A human agent will handle your request.


In [12]:
# Example 2: Low-value query (still goes through, but can be auto-approved later)
user_question = "What's my account balance?"
final_state = graph.invoke({"user_input": user_question})
print("\n--- FINAL OUTPUT ---")
print(final_state["approved_reply"])


--- HUMAN APPROVAL NEEDED ---
AI draft reply: I can help with your request: What's my account balance?

--- FINAL OUTPUT ---
A human agent will handle your request.


# Approach -2


In [13]:
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage
from langgraph.graph import add_messages, StateGraph, END
from langchain_openai import ChatOpenAI

In [14]:
# -----------------------------
# 1. Define the state
# -----------------------------
# The workflow state is a dictionary that carries messages between nodes.
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# -----------------------------
# 2. Setup the LLM
# -----------------------------
import os
os.environ['OPENAI_API_KEY'] = 'sk-proj-xxxxxxxxxxxxxxxxxxxxxxx'

In [16]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [21]:
# -----------------------------
# 3. Define node names (for clarity)
# -----------------------------
GENERATE_POST = "generate_post"
GET_REVIEW_DECISION = "get_review_decision"
POST = "post"
COLLECT_FEEDBACK = "collect_feedback"

In [22]:
# -----------------------------
# 4. Define workflow nodes
# -----------------------------
def generate_post(state: State):
    """Use the LLM to generate a draft LinkedIn post from the user’s request."""
    return {
        "messages": [llm.invoke(state["messages"])]
    }


In [23]:
def get_review_decision(state: State):
    """Human-in-the-loop: Show the AI draft and ask if it should be posted."""
    post_content = state["messages"][-1].content

    print("\n📢 Current LinkedIn Post:\n")
    print(post_content)
    print("\n")

    decision = input("Post to LinkedIn? (yes/no): ")

    if decision.lower() == "yes":
        return POST  # Route workflow to post node
    else:
        return COLLECT_FEEDBACK  # Route workflow to feedback node

In [24]:
def post(state: State):
    """If approved, display the final post and mark it as published."""
    final_post = state["messages"][-1].content
    print("\n📢 Final LinkedIn Post:\n")
    print(final_post)
    print("\n✅ Post has been approved and is now live on LinkedIn!")

In [25]:
def collect_feedback(state: State):
    """If rejected, collect human feedback and pass it back as a message to improve the draft."""
    feedback = input("How can I improve this post? ")
    return {
        "messages": [HumanMessage(content=feedback)]
    }

In [26]:
# -----------------------------
# 5. Build the graph
# -----------------------------
graph = StateGraph(State)

In [27]:
graph.add_node(GENERATE_POST, generate_post)
graph.add_node(GET_REVIEW_DECISION, get_review_decision)
graph.add_node(COLLECT_FEEDBACK, collect_feedback)
graph.add_node(POST, post)

graph.set_entry_point(GENERATE_POST)

graph.add_conditional_edges(GENERATE_POST, get_review_decision)
graph.add_edge(POST, END)
graph.add_edge(COLLECT_FEEDBACK, GENERATE_POST)

In [28]:
app = graph.compile()

In [29]:
response = app.invoke({
    "messages": [HumanMessage(content="Write me a LinkedIn post on AI Agents taking over content creation")]
})


📢 Current LinkedIn Post:

🚀✨ Embracing the Future: AI Agents in Content Creation 🌟🤖

As we navigate the rapidly evolving digital landscape, one exciting development is the rise of AI agents in content creation. From drafting articles and generating marketing copy to brainstorming ideas and personalizing user experiences, these intelligent systems are transforming how we think about creativity and communication. 

But what does this mean for creators, marketers, and businesses? 

1️⃣ **Efficiency**: AI agents can streamline content production, allowing human creators to focus on strategic thinking and high-level creativity. Imagine generating a month’s worth of blog posts in a matter of hours! 

2️⃣ **Personalization**: With advanced algorithms, AI can tailor content to specific audiences, enhancing engagement and improving conversion rates. Companies can now deliver the right message to the right person at the right time.

3️⃣ **Data-Driven Insights**: AI can analyze vast amounts of d

In [30]:
print(response)

{'messages': [HumanMessage(content='Write me a LinkedIn post on AI Agents taking over content creation', additional_kwargs={}, response_metadata={}, id='9c347417-5ca6-46d8-b0c2-b14c61661725'), AIMessage(content="🚀✨ Embracing the Future: AI Agents in Content Creation 🌟🤖\n\nAs we navigate the rapidly evolving digital landscape, one exciting development is the rise of AI agents in content creation. From drafting articles and generating marketing copy to brainstorming ideas and personalizing user experiences, these intelligent systems are transforming how we think about creativity and communication. \n\nBut what does this mean for creators, marketers, and businesses? \n\n1️⃣ **Efficiency**: AI agents can streamline content production, allowing human creators to focus on strategic thinking and high-level creativity. Imagine generating a month’s worth of blog posts in a matter of hours! \n\n2️⃣ **Personalization**: With advanced algorithms, AI can tailor content to specific audiences, enhanc